In [3]:
import os
import glob
from pathlib import Path

OUTPUT_ROOT = r"E:\NETS\processed_xy_only"
QC_COLLECT_DIR = r"E:\NETS\all_QC_images"

def check_sample_integrity(sample_dir, sample_name):
    """检查五个项目，返回缺失列表"""
    t2_final = os.path.join(sample_dir, f"{sample_name}_T2_final.nii.gz")
    ce_final = os.path.join(sample_dir, f"{sample_name}_CE_final.nii.gz")
    overlay_qc = os.path.join(sample_dir, f"{sample_name}_overlay_QC.jpg")
    t2_slices_dir = os.path.join(sample_dir, "T2_slices")
    ce_slices_dir = os.path.join(sample_dir, "CE_slices")
    
    missing = []
    if not (os.path.exists(t2_final) and os.path.getsize(t2_final) > 0):
        missing.append("T2_final.nii.gz")
    if not (os.path.exists(ce_final) and os.path.getsize(ce_final) > 0):
        missing.append("CE_final.nii.gz")
    if not (os.path.exists(overlay_qc) and os.path.getsize(overlay_qc) > 0):
        missing.append("overlay_QC.jpg")
    if not (os.path.isdir(t2_slices_dir) and len(glob.glob(os.path.join(t2_slices_dir, "*.jpg"))) > 0):
        missing.append("T2_slices文件夹无jpg")
    if not (os.path.isdir(ce_slices_dir) and len(glob.glob(os.path.join(ce_slices_dir, "*.jpg"))) > 0):
        missing.append("CE_slices文件夹无jpg")
    return missing

# 收集QC图（仍然可以执行）
os.makedirs(QC_COLLECT_DIR, exist_ok=True)
sample_dirs = [d for d in Path(OUTPUT_ROOT).iterdir() if d.is_dir()]
copied = 0
for sample_dir in sample_dirs:
    sample_name = sample_dir.name
    qc_file = sample_dir / f"{sample_name}_overlay_QC.jpg"
    if qc_file.exists():
        dest = Path(QC_COLLECT_DIR) / f"{sample_name}_overlay_QC.jpg"
        import shutil
        shutil.copy2(qc_file, dest)
        copied += 1
print(f"已复制 {copied} 张QC图片到 {QC_COLLECT_DIR}")

# 检查所有样本的完整性
missing_list = []
for sample_dir in sample_dirs:
    sample_name = sample_dir.name
    missing = check_sample_integrity(str(sample_dir), sample_name)
    if missing:
        missing_list.append((sample_name, missing))
        print(f"❌ {sample_name} 缺失: {missing}")
    else:
        print(f"✅ {sample_name} 完整")

print(f"\n总计 {len(sample_dirs)} 个样本，完整样本 {len(sample_dirs)-len(missing_list)}，缺失样本 {len(missing_list)}")
if missing_list:
    print("缺失样本及其缺失文件：")
    for name, m in missing_list:
        print(f"  {name}: {m}")

已复制 168 张QC图片到 E:\NETS\all_QC_images
✅ CGGA_1001 完整
✅ CGGA_1004 完整
✅ CGGA_1006 完整
✅ CGGA_1007 完整
✅ CGGA_1008 完整
✅ CGGA_1011 完整
✅ CGGA_1015 完整
✅ CGGA_1017 完整
✅ CGGA_1027 完整
✅ CGGA_1030 完整
✅ CGGA_1032 完整
✅ CGGA_1035 完整
✅ CGGA_1051 完整
✅ CGGA_1058 完整
✅ CGGA_1079 完整
✅ CGGA_1086 完整
✅ CGGA_1087 完整
✅ CGGA_1131 完整
✅ CGGA_1144 完整
✅ CGGA_1147 完整
✅ CGGA_1158 完整
✅ CGGA_1160 完整
✅ CGGA_1171 完整
✅ CGGA_1195 完整
✅ CGGA_1198 完整
✅ CGGA_1211 完整
✅ CGGA_1212 完整
✅ CGGA_1226 完整
✅ CGGA_1282 完整
✅ CGGA_1303 完整
✅ CGGA_1307 完整
✅ CGGA_1314 完整
✅ CGGA_1317 完整
✅ CGGA_1318 完整
✅ CGGA_1338 完整
✅ CGGA_1339 完整
✅ CGGA_1340 完整
✅ CGGA_1350 完整
✅ CGGA_1361 完整
✅ CGGA_1365 完整
✅ CGGA_1368 完整
✅ CGGA_1377 完整
✅ CGGA_1382 完整
✅ CGGA_1408 完整
✅ CGGA_1409 完整
✅ CGGA_1413 完整
✅ CGGA_1417 完整
✅ CGGA_1418 完整
✅ CGGA_1422 完整
✅ CGGA_1446 完整
✅ CGGA_1455 完整
✅ CGGA_1467 完整
✅ CGGA_1469 完整
✅ CGGA_1474 完整
✅ CGGA_1481 完整
✅ CGGA_1486 完整
✅ CGGA_1488 完整
✅ CGGA_1494 完整
✅ CGGA_1497 完整
✅ CGGA_1498 完整
✅ CGGA_1503 完整
✅ CGGA_1521 完整
✅ CGGA_1526 完整
✅ CGGA_1531 完整
✅ C

In [5]:
import os
from pathlib import Path

# 缺失样本的文件夹名称和路径
missing_names = ['CGGA_256', 'CGGA_276', 'CGGA_342', 'CGGA_871']
data_root = Path(r"E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered")
output_root = r"E:\NETS\processed_xy_only"

for name in missing_names:
    folder = data_root / name
    if not folder.exists():
        print(f"警告：{folder} 不存在")
        continue
    ce = find_file_by_keywords(folder, ['CE_bet', 'ce_bet'])
    t2 = find_file_by_keywords(folder, ['T2_bet', 't2_bet'])
    if ce and t2:
        print(f"正在处理 {name} ...")
        try:
            process_sample(ce, t2, name, output_root)
            print(f"  {name} 处理完成")
        except Exception as e:
            print(f"  {name} 处理失败: {e}")
    else:
        print(f"  {name} 缺少 CE 或 T2 文件，跳过")

正在处理 CGGA_256 ...
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_256\T2_bet.nii.gz
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_256\CE_bet.nii.gz
   ⚠️ 跳过 CGGA_256: T2层数(24) ≠ CE层数(12)
  CGGA_256 处理完成
正在处理 CGGA_276 ...
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_276\T2_bet.nii.gz
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_276\CE_bet.nii.gz
   ⚠️ 跳过 CGGA_276: T2层数(24) ≠ CE层数(22)
  CGGA_276 处理完成
正在处理 CGGA_342 ...
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_342\T2_bet.nii.gz
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_342\CE_bet.nii.gz
  

In [7]:
import os
import numpy as np
import nibabel as nib
import ants
from pathlib import Path
import shutil

# 配置路径
DATA_ROOT = Path(r"E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered")
OUTPUT_ROOT = Path(r"E:\NETS\processed_xy_only")

TARGET_XY = (1.0, 1.0)
REG_TYPE = 'Affine'
DO_NORMALIZE = True

# 需要修复的样本列表（排除 CGGA_256）
fix_samples = ['CGGA_276', 'CGGA_342', 'CGGA_871']

# 确保辅助函数已定义（如果没有，需要重新运行之前定义函数的单元格）
# 这里假设 safe_read_ants, resample_xy_only, normalize_image,
# generate_overlay_qc, save_all_slices_as_jpg, find_file_by_keywords 等已经存在

def align_layer_count(img, target_layers, remove_from_top=True):
    """删除多余层，保留连续的区域"""
    data = img.numpy()
    curr = data.shape[2]
    if curr == target_layers:
        return img
    diff = curr - target_layers
    if remove_from_top:
        new_data = data[:, :, diff:]
    else:
        new_data = data[:, :, :-diff]
    return ants.from_numpy(new_data, origin=img.origin, spacing=img.spacing, direction=img.direction)

for sample_name in fix_samples:
    sample_dir = OUTPUT_ROOT / sample_name
    if sample_dir.exists():
        # 删除旧目录，重新生成
        shutil.rmtree(sample_dir)
    sample_dir.mkdir(parents=True, exist_ok=True)
    
    folder = DATA_ROOT / sample_name
    ce = find_file_by_keywords(folder, ['CE_bet', 'ce_bet'])
    t2 = find_file_by_keywords(folder, ['T2_bet', 't2_bet'])
    if not (ce and t2):
        print(f"{sample_name}: 缺少CE或T2文件，跳过")
        continue
    
    print(f"\n处理 {sample_name} ...")
    # 1. 只重采样XY
    t2_resampled, _ = resample_xy_only(t2, str(sample_dir), f"{sample_name}_T2")
    ce_resampled, _ = resample_xy_only(ce, str(sample_dir), f"{sample_name}_CE_orig")
    
    # 2. 配准CE到T2空间
    reg = ants.registration(fixed=t2_resampled, moving=ce_resampled, type_of_transform=REG_TYPE)
    ce_warped = reg['warpedmovout']
    
    # 3. 强制层数一致：以T2层数为基准，裁剪CE的层数（删除多余层）
    t2_layers = t2_resampled.shape[2]
    ce_layers = ce_warped.shape[2]
    print(f"  原始 T2层数={t2_layers}, CE层数={ce_layers}")
    if ce_layers > t2_layers:
        # 删除CE的多余层（这里删除最上层，即头顶）
        ce_warped = align_layer_count(ce_warped, t2_layers, remove_from_top=True)
        print(f"  删除CE {ce_layers - t2_layers} 层，现在层数={t2_layers}")
    elif ce_layers < t2_layers:
        # 如果CE层数少，则删除T2的多余层（但通常T2是基准，不推荐；这里统一处理：删除T2的最上层）
        t2_resampled = align_layer_count(t2_resampled, ce_layers, remove_from_top=True)
        print(f"  删除T2 {t2_layers - ce_layers} 层，现在层数={ce_layers}")
    else:
        print("  层数已一致")
    
    # 4. 标准化
    if DO_NORMALIZE:
        t2_final = normalize_image(t2_resampled)
        ce_final = normalize_image(ce_warped)
    else:
        t2_final = t2_resampled
        ce_final = ce_warped
    
    # 5. 保存最终NIfTI
    t2_out = sample_dir / f"{sample_name}_T2_final.nii.gz"
    ce_out = sample_dir / f"{sample_name}_CE_final.nii.gz"
    ants.image_write(t2_final, str(t2_out))
    ants.image_write(ce_final, str(ce_out))
    
    # 6. 生成QC叠加图
    qc_path = sample_dir / f"{sample_name}_overlay_QC.jpg"
    generate_overlay_qc(t2_final, ce_final, str(qc_path))
    
    # 7. 生成所有切片JPG
    save_all_slices_as_jpg(t2_final, str(sample_dir), f"{sample_name}_T2_final", "T2")
    save_all_slices_as_jpg(ce_final, str(sample_dir), f"{sample_name}_CE_final", "CE")
    
    print(f"完成 {sample_name}")

print("修复完成")


处理 CGGA_276 ...
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_276\T2_bet.nii.gz
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_276\CE_bet.nii.gz
  原始 T2层数=24, CE层数=24
  层数已一致
   已保存 24 张 T2 图层
   已保存 24 张 CE 图层
完成 CGGA_276

处理 CGGA_342 ...
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_342\T2_bet.nii.gz
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_342\CE_bet.nii.gz
  原始 T2层数=24, CE层数=24
  层数已一致
   已保存 24 张 T2 图层
   已保存 24 张 CE 图层
完成 CGGA_342

处理 CGGA_871 ...
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_871\T2_bet.nii.gz
   ants.image_read失败，使用nibabel中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA

In [4]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
最终批量预处理脚本（完整版）：
- 重采样 XY 到 1mm，保留 Z 轴原始层厚
- 双向层数对齐（裁剪层数多的序列）
- 以指定参考样本（CGGA_P179）为固定图像，仿射配准
- Z-score 标准化
- 自动裁剪到脑组织边界框 + 应用脑组织掩膜（彻底清除所有背景，包括四角残余）
- 输出 NIfTI + 所有轴向切片 JPG
"""

import os
import sys
import numpy as np
import nibabel as nib
import ants
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from scipy import ndimage

# ==================== 用户配置 ====================
DATA_ROOT = r"E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered"
OUTPUT_ROOT = r"E:\NETS\processed_aligned"

REF_SAMPLE = 'CGGA_P179'
EXCLUDE_SAMPLES = ['CGGA_256']

CE_KEYWORDS = ['CE_bet', 'ce_bet']
T2_KEYWORDS = ['T2_bet', 't2_bet']

TARGET_XY = (1.0, 1.0)
REG_TYPE = 'Affine'
DO_NORMALIZE = True
SAVE_SLICES = True
REMOVE_WHERE = 'top'          # 层数对齐时删除顶层或底层

# 背景处理：裁剪边界框并应用掩膜（彻底清除背景）
CROP_AND_MASK = True
CROP_MARGIN = 20               # 边界框扩展体素数（避免切脑组织）

FIXED_XY_SIZE = None

# ==================== 辅助函数 ====================
def safe_read_ants(path):
    try:
        return ants.image_read(path)
    except:
        nib_img = nib.load(path)
        temp_dir = Path(os.environ.get("TEMP", "."))
        temp_path = temp_dir / f"temp_ants_{Path(path).stem}.nii"
        nib.save(nib_img, temp_path)
        ants_img = ants.image_read(str(temp_path))
        temp_path.unlink()
        return ants_img

def find_file_by_keywords(folder_path, keywords):
    folder = Path(folder_path)
    for ext in ['*.nii.gz', '*.nii']:
        for file in folder.glob(ext):
            if any(kw.lower() in file.name.lower() for kw in keywords):
                return str(file)
    return None

def resample_xy_only(in_path, out_dir, prefix):
    img = safe_read_ants(in_path)
    orig_spacing = img.spacing
    new_spacing = (TARGET_XY[0], TARGET_XY[1], orig_spacing[2])
    resampled = ants.resample_image(img, new_spacing, use_voxels=False, interp_type=1)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{prefix}_resampled_xy.nii.gz")
    ants.image_write(resampled, out_path)
    return resampled, out_path

def align_layers(t2_img, ce_img, remove_where='top'):
    t2_layers = t2_img.shape[2]
    ce_layers = ce_img.shape[2]
    if t2_layers == ce_layers:
        return t2_img, ce_img, t2_layers
    elif t2_layers > ce_layers:
        diff = t2_layers - ce_layers
        data = t2_img.numpy()
        if remove_where == 'top':
            new_data = data[:, :, diff:]
        else:
            new_data = data[:, :, :-diff]
        t2_cropped = ants.from_numpy(new_data, origin=t2_img.origin,
                                     spacing=t2_img.spacing,
                                     direction=t2_img.direction)
        print(f"   T2 层数({t2_layers}) > CE({ce_layers})，删除 T2 {diff} 层，最终层数={ce_layers}")
        return t2_cropped, ce_img, ce_layers
    else:
        diff = ce_layers - t2_layers
        data = ce_img.numpy()
        if remove_where == 'top':
            new_data = data[:, :, diff:]
        else:
            new_data = data[:, :, :-diff]
        ce_cropped = ants.from_numpy(new_data, origin=ce_img.origin,
                                     spacing=ce_img.spacing,
                                     direction=ce_img.direction)
        print(f"   CE 层数({ce_layers}) > T2({t2_layers})，删除 CE {diff} 层，最终层数={t2_layers}")
        return t2_img, ce_cropped, t2_layers

def crop_and_mask(t2_img, ce_img, margin=15):
    """
    改进版：使用1%阈值 + 形态学膨胀，减少脑组织误删
    """
    from scipy import ndimage
    data = t2_img.numpy()
    
    # 使用更低的阈值（1%分位数）
    thresh = np.percentile(data, 1)
    mask = data > thresh
    
    # 连通域分析
    labeled, num = ndimage.label(mask)
    if num == 0:
        raise ValueError("未检测到脑组织")
    sizes = ndimage.sum(mask, labeled, range(1, num+1))
    largest_label = np.argmax(sizes) + 1
    brain_mask = (labeled == largest_label)
    
    # 形态学膨胀（扩张脑组织区域）
    struct = ndimage.generate_binary_structure(3, 1)
    brain_mask = ndimage.binary_dilation(brain_mask, structure=struct, iterations=2)
    
    # 获取边界框
    coords = np.where(brain_mask)
    x_min, x_max = coords[0].min(), coords[0].max()
    y_min, y_max = coords[1].min(), coords[1].max()
    z_min, z_max = coords[2].min(), coords[2].max()
    x_min = max(0, x_min - margin)
    x_max = min(data.shape[0]-1, x_max + margin)
    y_min = max(0, y_min - margin)
    y_max = min(data.shape[1]-1, y_max + margin)
    z_min = max(0, z_min - margin)
    z_max = min(data.shape[2]-1, z_max + margin)
    
    # 裁剪图像
    t2_cropped_data = data[x_min:x_max+1, y_min:y_max+1, z_min:z_max+1]
    new_origin_t2 = (t2_img.origin[0] + x_min * t2_img.spacing[0],
                     t2_img.origin[1] + y_min * t2_img.spacing[1],
                     t2_img.origin[2] + z_min * t2_img.spacing[2])
    t2_cropped = ants.from_numpy(t2_cropped_data,
                                 origin=new_origin_t2,
                                 spacing=t2_img.spacing,
                                 direction=t2_img.direction)
    
    ce_data = ce_img.numpy()
    ce_cropped_data = ce_data[x_min:x_max+1, y_min:y_max+1, z_min:z_max+1]
    new_origin_ce = (ce_img.origin[0] + x_min * ce_img.spacing[0],
                     ce_img.origin[1] + y_min * ce_img.spacing[1],
                     ce_img.origin[2] + z_min * ce_img.spacing[2])
    ce_cropped = ants.from_numpy(ce_cropped_data,
                                 origin=new_origin_ce,
                                 spacing=ce_img.spacing,
                                 direction=ce_img.direction)
    
    # 在裁剪后图像上重新生成精确掩膜（同样使用低阈值+膨胀）
    cropped_data = t2_cropped.numpy()
    thresh2 = np.percentile(cropped_data, 1)
    mask2 = cropped_data > thresh2
    labeled2, num2 = ndimage.label(mask2)
    if num2 == 0:
        raise ValueError("裁剪后无脑组织")
    sizes2 = ndimage.sum(mask2, labeled2, range(1, num2+1))
    largest_label2 = np.argmax(sizes2) + 1
    final_mask = (labeled2 == largest_label2)
    final_mask = ndimage.binary_dilation(final_mask, structure=struct, iterations=3)
    final_mask = final_mask.astype(cropped_data.dtype)
    
    t2_masked_data = cropped_data * final_mask
    ce_masked_data = ce_cropped.numpy() * final_mask
    
    t2_masked = ants.from_numpy(t2_masked_data,
                                origin=t2_cropped.origin,
                                spacing=t2_cropped.spacing,
                                direction=t2_cropped.direction)
    ce_masked = ants.from_numpy(ce_masked_data,
                                origin=ce_cropped.origin,
                                spacing=ce_cropped.spacing,
                                direction=ce_cropped.direction)
    return t2_masked, ce_masked

def normalize_image(img):
    data = img.numpy()
    data = np.nan_to_num(data)
    mean, std = data.mean(), data.std()
    if std > 0:
        data = (data - mean) / std
    return ants.from_numpy(data, origin=img.origin, spacing=img.spacing, direction=img.direction)

def generate_overlay_qc(t2_img, ce_img, output_jpg):
    t2_data = t2_img.numpy()
    ce_data = ce_img.numpy()
    mid = t2_data.shape[2] // 2
    t2_slice = t2_data[:, :, mid].T
    ce_slice = ce_data[:, :, mid].T
    t2_norm = (t2_slice - t2_slice.min()) / (t2_slice.max() - t2_slice.min() + 1e-8)
    ce_norm = (ce_slice - ce_slice.min()) / (ce_slice.max() - ce_slice.min() + 1e-8)
    overlay = np.stack([t2_norm, ce_norm, np.zeros_like(t2_norm)], axis=-1)
    plt.figure(figsize=(6,6))
    plt.imshow(overlay)
    plt.axis('off')
    plt.savefig(output_jpg, bbox_inches='tight', pad_inches=0, dpi=150)
    plt.close()

def save_all_slices_as_jpg(img, base_dir, prefix, modality):
    data = img.numpy()
    n_slices = data.shape[2]
    slice_dir = os.path.join(base_dir, f"{modality}_slices")
    os.makedirs(slice_dir, exist_ok=True)
    data_min, data_max = data.min(), data.max()
    if data_max - data_min == 0:
        data_norm = np.zeros_like(data)
    else:
        data_norm = (data - data_min) / (data_max - data_min)
    for z in range(n_slices):
        slice_2d = data_norm[:, :, z].T
        plt.figure(figsize=(6,6))
        plt.imshow(slice_2d, cmap='gray', vmin=0, vmax=1)
        plt.axis('off')
        jpg_name = f"{prefix}_slice_{z:03d}.jpg"
        jpg_path = os.path.join(slice_dir, jpg_name)
        plt.savefig(jpg_path, bbox_inches='tight', pad_inches=0, dpi=150)
        plt.close()
    print(f"   已保存 {n_slices} 张 {modality} 图层")

def process_sample(ce_path, t2_path, sample_name, output_root, fixed_image, is_reference=False):
    sample_dir = os.path.join(output_root, sample_name)
    os.makedirs(sample_dir, exist_ok=True)
    print(f"\n处理样本: {sample_name}")

    # 1. XY 重采样
    t2_resampled, _ = resample_xy_only(t2_path, sample_dir, f"{sample_name}_T2")
    ce_resampled, _ = resample_xy_only(ce_path, sample_dir, f"{sample_name}_CE_orig")
    print(f"   T2 原始层数: {t2_resampled.shape[2]}, CE 原始层数: {ce_resampled.shape[2]}")

    # 2. 层数对齐
    t2_aligned, ce_aligned, final_layers = align_layers(t2_resampled, ce_resampled, REMOVE_WHERE)
    print(f"   对齐后层数: {final_layers}")

    if is_reference:
        t2_trans = t2_aligned
        ce_trans = ce_aligned
    else:
        reg = ants.registration(fixed=fixed_image, moving=t2_aligned, type_of_transform=REG_TYPE)
        t2_trans = ants.apply_transforms(fixed=fixed_image, moving=t2_aligned,
                                         transformlist=reg['fwdtransforms'],
                                         reference_image=t2_aligned, interpolator='linear')
        ce_trans = ants.apply_transforms(fixed=fixed_image, moving=ce_aligned,
                                         transformlist=reg['fwdtransforms'],
                                         reference_image=t2_aligned, interpolator='linear')

    if FIXED_XY_SIZE is not None:
        t2_trans = ants.resample_image(t2_trans, FIXED_XY_SIZE, use_voxels=False, interp_type=1)
        ce_trans = ants.resample_image(ce_trans, FIXED_XY_SIZE, use_voxels=False, interp_type=1)

    if DO_NORMALIZE:
        t2_norm = normalize_image(t2_trans)
        ce_norm = normalize_image(ce_trans)
    else:
        t2_norm = t2_trans
        ce_norm = ce_trans

    # 3. 背景裁剪 + 掩膜（彻底清除四角残余）
    if CROP_AND_MASK:
        try:
            t2_final, ce_final = crop_and_mask(t2_norm, ce_norm, margin=CROP_MARGIN)
            print(f"   背景裁剪+掩膜后尺寸: {t2_final.shape}")
        except Exception as e:
            print(f"   背景处理失败，使用原始尺寸: {e}")
            t2_final, ce_final = t2_norm, ce_norm
    else:
        t2_final, ce_final = t2_norm, ce_norm

    # 4. 保存 NIfTI
    t2_out = os.path.join(sample_dir, f"{sample_name}_T2_final.nii.gz")
    ce_out = os.path.join(sample_dir, f"{sample_name}_CE_final.nii.gz")
    ants.image_write(t2_final, t2_out)
    ants.image_write(ce_final, ce_out)

    # 5. QC 图
    qc_path = os.path.join(sample_dir, f"{sample_name}_overlay_QC.jpg")
    generate_overlay_qc(t2_final, ce_final, qc_path)

    # 6. 切片 JPG
    if SAVE_SLICES:
        save_all_slices_as_jpg(t2_final, sample_dir, f"{sample_name}_T2_final", "T2")
        save_all_slices_as_jpg(ce_final, sample_dir, f"{sample_name}_CE_final", "CE")

    print(f"✓ 完成 {sample_name}")
    return True

# ==================== 主流程 ====================
if __name__ == "__main__":
    print("="*70)
    print("批量 MRI 预处理（最终完整版）：层数对齐 + 仿射配准 + 背景裁剪+掩膜")
    print("参考样本: %s" % REF_SAMPLE)
    print("="*70)

    root = Path(DATA_ROOT)
    all_folders = [f for f in root.iterdir() if f.is_dir()]
    sample_infos = []
    for folder in all_folders:
        if folder.name in EXCLUDE_SAMPLES:
            print(f"⚠️ 跳过排除样本: {folder.name}")
            continue
        ce = find_file_by_keywords(folder, CE_KEYWORDS)
        t2 = find_file_by_keywords(folder, T2_KEYWORDS)
        if ce and t2:
            sample_infos.append((ce, t2, folder.name))
        else:
            print(f"⚠️ 跳过 {folder.name}: 缺少 CE 或 T2")
    print(f"共找到 {len(sample_infos)} 个有效样本")

    if not sample_infos:
        sys.exit(1)

    # 参考样本
    ref_info = None
    for ce, t2, name in sample_infos:
        if name == REF_SAMPLE:
            ref_info = (ce, t2, name)
            break
    if ref_info is None:
        print(f"❌ 参考样本 {REF_SAMPLE} 不存在")
        sys.exit(1)

    # 生成固定图像（参考样本的未裁剪 T2）
    ref_ce, ref_t2, ref_name = ref_info
    temp_dir = os.path.join(OUTPUT_ROOT, "temp")
    os.makedirs(temp_dir, exist_ok=True)
    t2_ref_resampled, _ = resample_xy_only(ref_t2, temp_dir, "ref_T2")
    ce_ref_resampled, _ = resample_xy_only(ref_ce, temp_dir, "ref_CE")
    t2_ref_aligned, _, _ = align_layers(t2_ref_resampled, ce_ref_resampled, REMOVE_WHERE)
    fixed_image = t2_ref_aligned
    print(f"参考样本 {ref_name} 固定图像尺寸: {fixed_image.shape}")

    # 处理参考样本本身
    print("\n处理参考样本...")
    process_sample(ref_ce, ref_t2, ref_name, OUTPUT_ROOT, fixed_image, is_reference=True)

    # 其他样本
    other_samples = [info for info in sample_infos if info[2] != REF_SAMPLE]
    print(f"\n即将处理其余 {len(other_samples)} 个样本")

    test_n = min(3, len(other_samples))
    print(f"\n🧪 测试前 {test_n} 个样本...")
    for i, (ce, t2, name) in enumerate(other_samples[:test_n]):
        print(f"\n--- 测试样本 {i+1}/{test_n}: {name} ---")
        try:
            success = process_sample(ce, t2, name, OUTPUT_ROOT, fixed_image, is_reference=False)
            if not success:
                print(f"❌ 样本 {name} 因层数不足被跳过")
                sys.exit(1)
        except Exception as e:
            print(f"❌ 测试失败: {e}")
            sys.exit(1)

    print(f"\n✅ 测试完成，请检查输出目录 {OUTPUT_ROOT} 中的 QC 图片")
    answer = input(f"\n是否继续处理全部 {len(other_samples)} 个样本？(y/n): ")
    if answer.lower() != 'y':
        print("已取消批量处理。")
        sys.exit(0)

    print(f"\n🚀 开始批量处理...")
    success_count = 0
    skip_count = 0
    for ce, t2, name in tqdm(other_samples, desc="批量处理进度"):
        try:
            ok = process_sample(ce, t2, name, OUTPUT_ROOT, fixed_image, is_reference=False)
            if ok:
                success_count += 1
            else:
                skip_count += 1
        except Exception as e:
            print(f"\n❌ {name} 错误: {e}")

    import shutil
    shutil.rmtree(temp_dir, ignore_errors=True)

    print("\n" + "="*70)
    print(f"批量处理完成！成功: {success_count}, 跳过: {skip_count}")
    print(f"所有结果保存在: {OUTPUT_ROOT}")
    print("="*70)

批量 MRI 预处理（最终完整版）：层数对齐 + 仿射配准 + 背景裁剪+掩膜
参考样本: CGGA_P179
⚠️ 跳过排除样本: CGGA_256
共找到 171 个有效样本
参考样本 CGGA_P179 固定图像尺寸: (186, 220, 24)

处理参考样本...

处理样本: CGGA_P179
   T2 原始层数: 24, CE 原始层数: 24
   对齐后层数: 24
   背景裁剪+掩膜后尺寸: (183, 211, 24)
   已保存 24 张 T2 图层
   已保存 24 张 CE 图层
✓ 完成 CGGA_P179

即将处理其余 170 个样本

🧪 测试前 3 个样本...

--- 测试样本 1/3: CGGA_1001 ---

处理样本: CGGA_1001
   T2 原始层数: 22, CE 原始层数: 22
   对齐后层数: 22
   背景裁剪+掩膜后尺寸: (186, 220, 24)
   已保存 24 张 T2 图层
   已保存 24 张 CE 图层
✓ 完成 CGGA_1001

--- 测试样本 2/3: CGGA_1004 ---

处理样本: CGGA_1004
   T2 原始层数: 22, CE 原始层数: 22
   对齐后层数: 22
   背景裁剪+掩膜后尺寸: (183, 216, 24)
   已保存 24 张 T2 图层
   已保存 24 张 CE 图层
✓ 完成 CGGA_1004

--- 测试样本 3/3: CGGA_1006 ---

处理样本: CGGA_1006
   T2 原始层数: 22, CE 原始层数: 22
   对齐后层数: 22
   背景裁剪+掩膜后尺寸: (186, 215, 24)
   已保存 24 张 T2 图层
   已保存 24 张 CE 图层
✓ 完成 CGGA_1006

✅ 测试完成，请检查输出目录 E:\NETS\processed_aligned 中的 QC 图片



是否继续处理全部 170 个样本？(y/n):  Y



🚀 开始批量处理...


批量处理进度:   0%|          | 0/170 [00:00<?, ?it/s]


处理样本: CGGA_1001
   T2 原始层数: 22, CE 原始层数: 22
   对齐后层数: 22
   背景裁剪+掩膜后尺寸: (186, 220, 24)
   已保存 24 张 T2 图层


...

   已保存 24 张 CE 图层
✓ 完成 CGGA_P86

处理样本: CGGA_P89
   T2 原始层数: 23, CE 原始层数: 23
   对齐后层数: 23
   背景裁剪+掩膜后尺寸: (186, 218, 24)
   已保存 24 张 T2 图层


批量处理进度:  99%|█████████▉| 169/170 [23:04<00:07,  7.66s/it]

   已保存 24 张 CE 图层
✓ 完成 CGGA_P89

处理样本: CGGA_P93
   T2 原始层数: 24, CE 原始层数: 24
   对齐后层数: 24
   背景裁剪+掩膜后尺寸: (185, 218, 24)
   已保存 24 张 T2 图层


批量处理进度: 100%|██████████| 170/170 [23:11<00:00,  8.19s/it]

   已保存 24 张 CE 图层
✓ 完成 CGGA_P93

批量处理完成！成功: 170, 跳过: 0
所有结果保存在: E:\NETS\processed_aligned


In [9]:
###特殊病历单独处理
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
单独处理样本：CGGA_871, CGGA_276
- 以 CGGA_P179 为参考，仿射配准
- 允许裁剪 T2 或 CE 多余层，使两个序列层数一致
- 输出 NIfTI、轴向切片 JPG、QC 图
- 收集 QC 图到单独文件夹
"""

import os
import sys
import numpy as np
import nibabel as nib
import ants
import matplotlib.pyplot as plt
from pathlib import Path
import shutil

# ==================== 用户配置 ====================
DATA_ROOT = r"E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered"
OUTPUT_ROOT = r"E:\NETS\processed_selected"
QC_COLLECT_DIR = os.path.join(OUTPUT_ROOT, "QC_collection")

SAMPLE_LIST = ['CGGA_871', 'CGGA_276']
REF_SAMPLE = 'CGGA_P179'

CE_KEYWORDS = ['CE_bet', 'ce_bet']
T2_KEYWORDS = ['T2_bet', 't2_bet']

TARGET_XY = (1.0, 1.0)
REG_TYPE = 'Affine'
DO_NORMALIZE = True
SAVE_SLICES = True
REMOVE_WHERE = 'top'          # 删除最上层或最下层
ALLOW_CROP_T2 = True          # 允许裁剪 T2 以匹配 CE（当 CE 层数少时）

FIXED_XY_SIZE = None

# ==================== 辅助函数 ====================
def safe_read_ants(path):
    try:
        return ants.image_read(path)
    except Exception as e:
        print(f"   ants.image_read 失败，使用 nibabel 中转: {e}")
        nib_img = nib.load(path)
        temp_dir = Path(os.environ.get("TEMP", "."))
        temp_path = temp_dir / f"temp_ants_{Path(path).stem}.nii"
        nib.save(nib_img, temp_path)
        ants_img = ants.image_read(str(temp_path))
        temp_path.unlink()
        return ants_img

def find_file_by_keywords(folder_path, keywords):
    folder = Path(folder_path)
    for ext in ['*.nii.gz', '*.nii']:
        for file in folder.glob(ext):
            if any(kw.lower() in file.name.lower() for kw in keywords):
                return str(file)
    return None

def resample_xy_only(in_path, out_dir, prefix):
    img = safe_read_ants(in_path)
    orig_spacing = img.spacing
    new_spacing = (TARGET_XY[0], TARGET_XY[1], orig_spacing[2])
    resampled = ants.resample_image(img, new_spacing, use_voxels=False, interp_type=1)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{prefix}_resampled_xy.nii.gz")
    ants.image_write(resampled, out_path)
    return resampled, out_path

def align_layers(t2_img, ce_img, remove_where='top', allow_crop_t2=False):
    t2_layers = t2_img.shape[2]
    ce_layers = ce_img.shape[2]
    if t2_layers == ce_layers:
        return t2_img, ce_img, t2_layers
    elif ce_layers > t2_layers:
        diff = ce_layers - t2_layers
        data = ce_img.numpy()
        if remove_where == 'top':
            new_data = data[:, :, diff:]
        else:
            new_data = data[:, :, :-diff]
        ce_cropped = ants.from_numpy(new_data, origin=ce_img.origin,
                                     spacing=ce_img.spacing,
                                     direction=ce_img.direction)
        print(f"   CE 层数({ce_layers}) > T2({t2_layers})，删除 CE {diff} 层（{'最上层' if remove_where=='top' else '最下层'}）")
        return t2_img, ce_cropped, t2_layers
    else:  # ce_layers < t2_layers
        if not allow_crop_t2:
            print(f"   ⚠️ CE层数({ce_layers}) < T2层数({t2_layers})，且不允许裁剪T2，跳过")
            return None, None, None
        diff = t2_layers - ce_layers
        data = t2_img.numpy()
        if remove_where == 'top':
            new_data = data[:, :, diff:]
        else:
            new_data = data[:, :, :-diff]
        t2_cropped = ants.from_numpy(new_data, origin=t2_img.origin,
                                     spacing=t2_img.spacing,
                                     direction=t2_img.direction)
        print(f"   CE 层数({ce_layers}) < T2({t2_layers})，删除 T2 {diff} 层（{'最上层' if remove_where=='top' else '最下层'}），最终层数 = {ce_layers}")
        return t2_cropped, ce_img, ce_layers

def normalize_image(img):
    data = img.numpy()
    data = np.nan_to_num(data)
    mean, std = data.mean(), data.std()
    if std > 0:
        data = (data - mean) / std
    return ants.from_numpy(data, origin=img.origin, spacing=img.spacing, direction=img.direction)

def generate_overlay_qc(t2_img, ce_img, output_jpg):
    t2_data = t2_img.numpy()
    ce_data = ce_img.numpy()
    mid = t2_data.shape[2] // 2
    t2_slice = t2_data[:, :, mid].T
    ce_slice = ce_data[:, :, mid].T
    t2_norm = (t2_slice - t2_slice.min()) / (t2_slice.max() - t2_slice.min() + 1e-8)
    ce_norm = (ce_slice - ce_slice.min()) / (ce_slice.max() - ce_slice.min() + 1e-8)
    overlay = np.stack([t2_norm, ce_norm, np.zeros_like(t2_norm)], axis=-1)
    plt.figure(figsize=(6,6))
    plt.imshow(overlay)
    plt.axis('off')
    plt.savefig(output_jpg, bbox_inches='tight', pad_inches=0, dpi=150)
    plt.close()

def save_all_slices_as_jpg(img, base_dir, prefix, modality):
    data = img.numpy()
    n_slices = data.shape[2]
    slice_dir = os.path.join(base_dir, f"{modality}_slices")
    os.makedirs(slice_dir, exist_ok=True)
    data_min, data_max = data.min(), data.max()
    if data_max - data_min == 0:
        data_norm = np.zeros_like(data)
    else:
        data_norm = (data - data_min) / (data_max - data_min)
    for z in range(n_slices):
        slice_2d = data_norm[:, :, z].T
        plt.figure(figsize=(6,6))
        plt.imshow(slice_2d, cmap='gray', vmin=0, vmax=1)
        plt.axis('off')
        jpg_name = f"{prefix}_slice_{z:03d}.jpg"
        jpg_path = os.path.join(slice_dir, jpg_name)
        plt.savefig(jpg_path, bbox_inches='tight', pad_inches=0, dpi=150)
        plt.close()
    print(f"   已保存 {n_slices} 张 {modality} 图层")

def process_sample(ce_path, t2_path, sample_name, output_root, fixed_image=None, is_reference=False):
    sample_dir = os.path.join(output_root, sample_name)
    os.makedirs(sample_dir, exist_ok=True)
    print(f"\n处理样本: {sample_name}")

    # 1. XY 重采样
    t2_resampled, _ = resample_xy_only(t2_path, sample_dir, f"{sample_name}_T2")
    ce_resampled, _ = resample_xy_only(ce_path, sample_dir, f"{sample_name}_CE_orig")

    print(f"   T2 原始层数: {t2_resampled.shape[2]}, CE 原始层数: {ce_resampled.shape[2]}")

    # 2. 层数对齐
    t2_aligned, ce_aligned, final_layers = align_layers(t2_resampled, ce_resampled, REMOVE_WHERE, ALLOW_CROP_T2)
    if t2_aligned is None:
        return None
    print(f"   对齐后层数: {final_layers}")

    if is_reference:
        t2_trans = t2_aligned
        ce_trans = ce_aligned
    else:
        # 配准到固定图像
        reg = ants.registration(fixed=fixed_image, moving=t2_aligned, type_of_transform=REG_TYPE)
        t2_trans = ants.apply_transforms(
            fixed=fixed_image,
            moving=t2_aligned,
            transformlist=reg['fwdtransforms'],
            reference_image=t2_aligned,
            interpolator='linear'
        )
        ce_trans = ants.apply_transforms(
            fixed=fixed_image,
            moving=ce_aligned,
            transformlist=reg['fwdtransforms'],
            reference_image=t2_aligned,
            interpolator='linear'
        )

    if FIXED_XY_SIZE is not None:
        t2_trans = ants.resample_image(t2_trans, FIXED_XY_SIZE, use_voxels=False, interp_type=1)
        ce_trans = ants.resample_image(ce_trans, FIXED_XY_SIZE, use_voxels=False, interp_type=1)

    if DO_NORMALIZE:
        t2_final = normalize_image(t2_trans)
        ce_final = normalize_image(ce_trans)
    else:
        t2_final = t2_trans
        ce_final = ce_trans

    t2_out = os.path.join(sample_dir, f"{sample_name}_T2_final.nii.gz")
    ce_out = os.path.join(sample_dir, f"{sample_name}_CE_final.nii.gz")
    ants.image_write(t2_final, t2_out)
    ants.image_write(ce_final, ce_out)

    qc_path = os.path.join(sample_dir, f"{sample_name}_overlay_QC.jpg")
    generate_overlay_qc(t2_final, ce_final, qc_path)

    if SAVE_SLICES:
        save_all_slices_as_jpg(t2_final, sample_dir, f"{sample_name}_T2_final", "T2")
        save_all_slices_as_jpg(ce_final, sample_dir, f"{sample_name}_CE_final", "CE")

    print(f"✓ 完成 {sample_name}")
    return qc_path

# ==================== 主流程 ====================
if __name__ == "__main__":
    print("="*70)
    print("单独处理样本: %s" % ", ".join(SAMPLE_LIST))
    print("参考样本: %s" % REF_SAMPLE)
    print("层数调整策略: 允许裁剪 T2 或 CE，以较小层数为准，删除最上层")
    print("="*70)

    # 获取所有样本文件路径
    root = Path(DATA_ROOT)
    sample_file_map = {}
    for folder in root.iterdir():
        if not folder.is_dir():
            continue
        ce = find_file_by_keywords(folder, CE_KEYWORDS)
        t2 = find_file_by_keywords(folder, T2_KEYWORDS)
        if ce and t2:
            sample_file_map[folder.name] = (ce, t2)

    if REF_SAMPLE not in sample_file_map:
        print(f"❌ 参考样本 {REF_SAMPLE} 不存在")
        sys.exit(1)

    # 先处理参考样本本身（使其成为固定图像）
    ref_ce, ref_t2 = sample_file_map[REF_SAMPLE]
    temp_dir = os.path.join(OUTPUT_ROOT, "temp")
    os.makedirs(temp_dir, exist_ok=True)
    # 对参考样本进行重采样和层数对齐
    t2_ref_resampled, _ = resample_xy_only(ref_t2, temp_dir, "ref_T2")
    ce_ref_resampled, _ = resample_xy_only(ref_ce, temp_dir, "ref_CE")
    t2_ref, ce_ref, _ = align_layers(t2_ref_resampled, ce_ref_resampled, REMOVE_WHERE, ALLOW_CROP_T2)
    if t2_ref is None:
        print("❌ 参考样本自身层数无法对齐")
        sys.exit(1)
    fixed_image = t2_ref   # 使用对齐后的 T2 作为固定图像
    print(f"参考样本 {REF_SAMPLE} 对齐后尺寸: {fixed_image.shape}, spacing: {fixed_image.spacing}")

    # 处理目标样本
    os.makedirs(QC_COLLECT_DIR, exist_ok=True)
    for sample_name in SAMPLE_LIST:
        if sample_name not in sample_file_map:
            print(f"⚠️ 样本 {sample_name} 不存在，跳过")
            continue
        ce_path, t2_path = sample_file_map[sample_name]
        try:
            qc_path = process_sample(ce_path, t2_path, sample_name, OUTPUT_ROOT, fixed_image, is_reference=False)
            if qc_path:
                dest = os.path.join(QC_COLLECT_DIR, f"{sample_name}_overlay_QC.jpg")
                shutil.copy2(qc_path, dest)
                print(f"   QC 图已复制到 {dest}")
        except Exception as e:
            print(f"❌ 处理 {sample_name} 出错: {e}")

    # 清理临时文件夹
    shutil.rmtree(temp_dir, ignore_errors=True)
    print("\n" + "="*70)
    print(f"处理完成！结果保存在: {OUTPUT_ROOT}")
    print(f"QC 图汇总在: {QC_COLLECT_DIR}")
    print("="*70)

单独处理样本: CGGA_871, CGGA_276
参考样本: CGGA_P179
层数调整策略: 允许裁剪 T2 或 CE，以较小层数为准，删除最上层
   ants.image_read 失败，使用 nibabel 中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_P179\T2_bet.nii.gz
   ants.image_read 失败，使用 nibabel 中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_P179\CE_bet.nii.gz
参考样本 CGGA_P179 对齐后尺寸: (186, 220, 24), spacing: (1.0, 1.0, 6.500002384185791)

处理样本: CGGA_871
   ants.image_read 失败，使用 nibabel 中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_871\T2_bet.nii.gz
   ants.image_read 失败，使用 nibabel 中转: Could not create ImageIO object for file E:\NETS\影像数据\原数据\1_CGGA.MRI_172_filtered\CGGA_871\CE_bet.nii.gz
   T2 原始层数: 24, CE 原始层数: 23
   CE 层数(23) < T2(24)，删除 T2 1 层（最上层），最终层数 = 23
   对齐后层数: 23
   已保存 24 张 T2 图层
   已保存 24 张 CE 图层
✓ 完成 CGGA_871
   QC 图已复制到 E:\NETS\processed_selected\QC_collection\CGGA_871_overlay_QC.jpg

处理样本: CGGA_276
   ants.image_read 失败，使用 nib

In [13]:
import os
import shutil
from pathlib import Path

# 配置
OUTPUT_ROOT = r"E:\NETS\processed_aligned"     # 预处理输出根目录
QC_COLLECT_DIR = r"E:\NETS\all_qc_images"      # 收集QC图片的目标文件夹

# 创建目标文件夹
os.makedirs(QC_COLLECT_DIR, exist_ok=True)

# 遍历所有样本文件夹
sample_dirs = [d for d in Path(OUTPUT_ROOT).iterdir() if d.is_dir()]
copied = 0
for sample_dir in sample_dirs:
    sample_name = sample_dir.name
    qc_file = sample_dir / f"{sample_name}_overlay_QC.jpg"
    if qc_file.exists():
        dest = Path(QC_COLLECT_DIR) / f"{sample_name}_overlay_QC.jpg"
        shutil.copy2(qc_file, dest)
        copied += 1
        print(f"已复制: {sample_name}")
    else:
        print(f"⚠️ 样本 {sample_name} 缺少 QC 图片")

print(f"\n完成！共复制 {copied} 张 QC 图片到 {QC_COLLECT_DIR}")

已复制: CGGA_1001
已复制: CGGA_1004
已复制: CGGA_1006
已复制: CGGA_1007
已复制: CGGA_1008
已复制: CGGA_1011
已复制: CGGA_1015
已复制: CGGA_1017
已复制: CGGA_1027
已复制: CGGA_1030
已复制: CGGA_1032
已复制: CGGA_1035
已复制: CGGA_1051
已复制: CGGA_1058
已复制: CGGA_1079
已复制: CGGA_1086
已复制: CGGA_1087
已复制: CGGA_1131
已复制: CGGA_1144
已复制: CGGA_1147
已复制: CGGA_1158
已复制: CGGA_1160
已复制: CGGA_1171
已复制: CGGA_1195
已复制: CGGA_1198
已复制: CGGA_1211
已复制: CGGA_1212
已复制: CGGA_1226
已复制: CGGA_1282
已复制: CGGA_1303
已复制: CGGA_1307
已复制: CGGA_1314
已复制: CGGA_1317
已复制: CGGA_1318
已复制: CGGA_1338
已复制: CGGA_1339
已复制: CGGA_1340
已复制: CGGA_1350
已复制: CGGA_1361
已复制: CGGA_1365
已复制: CGGA_1368
已复制: CGGA_1377
已复制: CGGA_1382
已复制: CGGA_1408
已复制: CGGA_1409
已复制: CGGA_1413
已复制: CGGA_1417
已复制: CGGA_1418
已复制: CGGA_1422
已复制: CGGA_1446
已复制: CGGA_1455
已复制: CGGA_1467
已复制: CGGA_1469
已复制: CGGA_1474
已复制: CGGA_1481
已复制: CGGA_1486
已复制: CGGA_1488
已复制: CGGA_1494
已复制: CGGA_1497
已复制: CGGA_1498
已复制: CGGA_1503
已复制: CGGA_1521
已复制: CGGA_1526
已复制: CGGA_1531
已复制: CGGA_1536
已复制: CGGA_1542
已复制: CGGA_